# Урок 9.2. LightGBM: рост по листьям, GOSS и связывание признаков

**Интерактивная версия:** `lessons/lesson_9_2/web/index.html`

1. Рост по листьям в `gbcourse`: тот же бюджет листьев — меньше ошибка на обучении.
2. `num_leaves` и `min_child_samples` на маленьких данных.
3. GOSS своими руками: потеря выигрыша против равномерной подвыборки.
4. GOSS в LightGBM: время и качество.
5. EFB на разреженных one-hot признаках.

In [1]:
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "python" / "gbcourse").is_dir())
sys.path.insert(0, str(ROOT / "shared" / "python"))

import lightgbm as lgb
import numpy as np
import pandas as pd
import scipy.sparse as sp
from gbcourse import datasets, GBClassifier, GBRegressor, RegressionTree

## 1. Рост по уровням и по листьям

In [2]:
X, y = datasets.friedman1(n=1000, noise=1.0, seed=94)
X_tr, X_val, y_tr, y_val = datasets.train_test_split(X, y, test_size=0.3, seed=0)
g = y_tr.mean() - y_tr
rows = []
for d in (2, 3, 4, 5, 6):
    dw = RegressionTree(max_depth=d).fit(X_tr, g)
    lw = RegressionTree(max_depth=None, max_leaves=2**d, growth="leafwise").fit(X_tr, g)
    err = lambda t, X_, y_: 0.5 * np.mean((y_ - y_tr.mean() - t.predict(X_)) ** 2)
    rows.append({"листьев": 2**d, "по уровням: обуч.": err(dw, X_tr, y_tr), "по листьям: обуч.": err(lw, X_tr, y_tr),
                 "по уровням: валид.": err(dw, X_val, y_val), "по листьям: валид.": err(lw, X_val, y_val),
                 "глубина по листьям": lw.depth})
pd.DataFrame(rows).round(3)

,листьев,по уровням: обуч.,по листьям: обуч.,по уровням: валид.,по листьям: валид.,глубина по листьям
0,4,6.608,6.406,6.721,7.000,3
1,8,4.646,4.444,5.038,5.018,5
2,16,3.738,3.316,4.632,4.293,7
3,32,2.910,2.346,4.118,4.179,9
4,64,2.033,1.308,4.304,4.123,10


## 2. num_leaves и min_child_samples

In [3]:
rows = []
for leaves in (4, 8, 16, 31, 64, 128):
    row = {"num_leaves": leaves}
    for mcs in (20, 5):
        m = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=leaves, min_child_samples=mcs, verbose=-1)
        m.fit(X_tr, y_tr, eval_X=(X_val,), eval_y=(y_val,), callbacks=[lgb.early_stopping(50, verbose=False)])
        row[f"MSE, min_child_samples={mcs}"] = min(m.evals_result_["valid_0"]["l2"])
    rows.append(row)
pd.DataFrame(rows).round(3)

,num_leaves,"MSE, min_child_samples=20","MSE, min_child_samples=5"
0,4,1.843,1.808
1,8,2.061,1.983
2,16,2.144,2.032
3,31,2.292,2.497
4,64,2.292,2.786
5,128,2.292,3.119


При `min_child_samples=20` и 700 объектах в дереве не может быть больше 35 листьев — поэтому 31, 64 и 128 дают одно и то же.

## 3. GOSS своими руками

In [4]:
Xc, yc = datasets.classification_2d(kind="moons", n=2000, noise=0.25, seed=93)
order = np.argsort(Xc[:, 0], kind="stable")


def lost_gain(g, h, w, full):
    gw, hw = (g * w)[order], (h * w)[order]
    GL, HL = np.cumsum(gw)[:-1], np.cumsum(hw)[:-1]
    G, H = gw.sum(), hw.sum()
    c = 0.5 * (GL**2 / (HL + 1) + (G - GL) ** 2 / (H - HL + 1) - G**2 / (H + 1))
    k = int(np.argmax(full))
    return (full[k] - full[int(np.argmax(c))]) / full[k]


rng = np.random.default_rng(0)
a, b, n = 0.2, 0.1, len(yc)
rows = []
for m_iter in (5, 30, 100):
    p = GBClassifier(mode="newton", n_estimators=m_iter, learning_rate=0.3, max_depth=3).fit(Xc, yc).predict_proba(Xc)[:, 1]
    g, h = p - yc, p * (1 - p)
    ones = np.ones(n)
    gw, hw = (g * ones)[order], (h * ones)[order]
    GL, HL = np.cumsum(gw)[:-1], np.cumsum(hw)[:-1]
    full = 0.5 * (GL**2 / (HL + 1) + (g.sum() - GL) ** 2 / (h.sum() - HL + 1) - g.sum() ** 2 / (h.sum() + 1))
    top = np.argsort(-np.abs(g))[: int(a * n)]
    rest = np.setdiff1d(np.arange(n), top)
    lu, lg = [], []
    for _ in range(300):
        w = np.zeros(n)
        w[rng.choice(n, int((a + b) * n), replace=False)] = 1 / (a + b)
        lu.append(lost_gain(g, h, w, full))
        w = np.zeros(n)
        w[top] = 1
        w[rng.choice(rest, int(b * n), replace=False)] = (1 - a) / b
        lg.append(lost_gain(g, h, w, full))
    rows.append({"деревьев": m_iter, "доля Σ|g| у топ-20%": np.abs(g[top]).sum() / np.abs(g).sum(),
                 "потеря: равномерно, %": 100 * np.mean(lu), "потеря: GOSS, %": 100 * np.mean(lg)})
pd.DataFrame(rows).round(2)

,деревьев,доля Σ|g| у топ-20%,"потеря: равномерно, %","потеря: GOSS, %"
0,5,0.44,7.45,4.08
1,30,0.84,59.48,20.71
2,100,0.93,77.19,19.75


## 4. GOSS в LightGBM

Время зависит от машины.

In [5]:
Xb, yb = datasets.classification_2d(kind="moons", n=400_000, noise=0.3, seed=95)
Xb = np.column_stack([Xb, np.random.default_rng(0).normal(size=(len(yb), 48))])
a_, b_, c_, d_ = datasets.train_test_split(Xb, yb, test_size=0.25, seed=0)
rows = []
for name, kw in [("все строки", {}), ("GOSS", dict(data_sample_strategy="goss")),
                 ("равномерно 30%", dict(subsample=0.3, subsample_freq=1))]:
    t0 = time.perf_counter()
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, verbose=-1, n_jobs=4, **kw).fit(a_, c_)
    el = time.perf_counter() - t0
    p = np.clip(m.predict_proba(b_)[:, 1], 1e-15, 1 - 1e-15)
    rows.append({"вариант": name, "время, с": el, "log-loss на тесте": -np.mean(d_ * np.log(p) + (1 - d_) * np.log(1 - p))})
pd.DataFrame(rows).round(4)

,вариант,"время, с",log-loss на тесте
0,все строки,3.1644,0.2141
1,GOSS,3.1460,0.2264
2,равномерно 30%,3.5647,0.2620


## 5. EFB

In [6]:
rng = np.random.default_rng(0)
n_rows, K, card = 200_000, 5, 200
cats = rng.integers(0, card, size=(n_rows, K))
Xs = sp.csr_matrix((np.ones(n_rows * K), (np.repeat(np.arange(n_rows), K), (cats + np.arange(K) * card).ravel())),
                   shape=(n_rows, K * card))
ys = rng.normal(size=(K, card))[np.arange(K), cats].sum(1) + rng.normal(size=n_rows)
print("столбцов:", Xs.shape[1], "| ненулевых в строке:", K)
for eb in (True, False):
    t0 = time.perf_counter()
    m = lgb.LGBMRegressor(n_estimators=100, enable_bundle=eb, verbose=-1, n_jobs=4).fit(Xs, ys)
    print(f"enable_bundle={eb}: {time.perf_counter() - t0:.2f} с, MSE обучения {np.mean((ys - m.predict(Xs)) ** 2):.4f}")

столбцов: 1000 | ненулевых в строке: 5


enable_bundle=True: 0.81 с, MSE обучения 2.1586


enable_bundle=False: 0.99 с, MSE обучения 2.1586


## Упражнения

1. Повторите раздел 3 для регрессии (`regression_1d(kind="wave", n=1000, noise=0.3)`, квадратичная потеря, 10 деревьев).
   Сконцентрированы ли градиенты? Помогает ли GOSS?
2. Подберите `num_leaves` и `max_depth` так, чтобы LightGBM на данных раздела 2 выиграл у варианта с 4 листьями.

Решения: `python lessons/lesson_9_2/exercises/solutions.py`.